In [ ]:
import json
from pathlib import Path


def sequence_to_segments(seq, topic=""):
    """
    Convert a 0/1 sequence where 1 marks the start of a new segment
    into contiguous segments covering the whole sequence.

    Example:
        [1, 0, 0, 1, 0, 1, 0]
    ->
        [
            {"topic": "", "start_sentence": 0, "end_sentence": 2},
            {"topic": "", "start_sentence": 3, "end_sentence": 4},
            {"topic": "", "start_sentence": 5, "end_sentence": 6}
        ]
    """
    n = len(seq)
    if n == 0:
        return []

    starts = [i for i, x in enumerate(seq) if x == 1]

    if not starts:
        return [{
            "topic": topic,
            "start_sentence": 0,
            "end_sentence": n - 1
        }]

    if starts[0] != 0:
        starts = [0] + starts

    segments = []
    for i, start in enumerate(starts):
        end = starts[i + 1] - 1 if i + 1 < len(starts) else n - 1
        segments.append({
            "topic": topic,
            "start_sentence": start,
            "end_sentence": end
        })

    return segments


def save_json_per_file(input_path, output_dir, use_field="pred"):
    """
    Input format example:
    {
        "IS1008b": {"ref": [...], "pred": [...]},
        "IS1009c": {"ref": [...], "pred": [...]}
    }

    Output:
        output_dir/IS1008b.json
        output_dir/IS1009c.json
    """
    input_path = Path(input_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for file_name, item in data.items():
        if isinstance(item, dict):
            if use_field not in item:
                print(f"Skipping {file_name}: no '{use_field}' field")
                continue
            seq = item[use_field]
        elif isinstance(item, list):
            # supports format like {"IS1008b": [0,1,0,...]}
            seq = item
        else:
            print(f"Skipping {file_name}: unsupported format")
            continue

        segments = sequence_to_segments(seq)

        out_path = output_dir / f"{file_name}.json"
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(segments, f, indent=4)

        print(f"Saved {out_path}")


if __name__ == "__main__":
    input_file = "/dataset/integrated_result/compare_dict_llama3_8b_0shot_filtered.json"
    output_folder = "/dataset/ami/segmentation_output/AMI_GT"

    # use_field="pred" for predicted labels
    # change to use_field="ref" if needed
    save_json_per_file(input_file, output_folder, use_field="ref")

Saved /home/linyun/dynamic_summary/dataset/ami/segmentation_output/AMI_GT/IS1008b.json
Saved /home/linyun/dynamic_summary/dataset/ami/segmentation_output/AMI_GT/IS1009b.json
Saved /home/linyun/dynamic_summary/dataset/ami/segmentation_output/AMI_GT/IS1007c.json
Saved /home/linyun/dynamic_summary/dataset/ami/segmentation_output/AMI_GT/IS1001d.json
Saved /home/linyun/dynamic_summary/dataset/ami/segmentation_output/AMI_GT/IS1005a.json
Saved /home/linyun/dynamic_summary/dataset/ami/segmentation_output/AMI_GT/ES2003a.json
Saved /home/linyun/dynamic_summary/dataset/ami/segmentation_output/AMI_GT/IS1004b.json
Saved /home/linyun/dynamic_summary/dataset/ami/segmentation_output/AMI_GT/IS1004d.json
Saved /home/linyun/dynamic_summary/dataset/ami/segmentation_output/AMI_GT/IS1006b.json
Saved /home/linyun/dynamic_summary/dataset/ami/segmentation_output/AMI_GT/IS1007a.json
Saved /home/linyun/dynamic_summary/dataset/ami/segmentation_output/AMI_GT/IS1002b.json
Saved /home/linyun/dynamic_summary/dataset/